In [ ]:
import os
from typing import TypedDict,Annotated
from langchain_core.messages import HumanMessage
from langgraph.graph.message import BaseMessage,add_messages
from dotenv import load_dotenv
from langchain_huggingface import  HuggingFaceEndpointEmbeddings 
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langgraph.prebuilt import ToolNode , tools_condition 

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from langchain_groq import ChatGroq
from langchain_core.tools import tool

load_dotenv()
os.environ['LANGCHAIN_PROJECT']="langgraph_rag"

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b",
    temperature=0.7)

In [ ]:
pdf = PyPDFLoader(r"file_PATH")
pdf_data = pdf.load()
final_pdf_data = []
for doc in pdf_data:final_pdf_data.append(doc)
final_pdf_data

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 400)
chunks = text_splitter.split_documents(final_pdf_data)
len(chunks)
print(chunks[:1])

In [ ]:
embeddings = HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(documents=chunks , embedding=embeddings)
vector_db

In [ ]:

retriver = vector_db.as_retriever(search_type ='similarity',search_kwargs={"k":4})



In [ ]:

class graph_state(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [ ]:
graph = StateGraph(graph_state)

In [ ]:
@tool()
def rag_Tool(query:str)->dict:
    '''Search the user's resume PDF (projects, skills, education, certifications) and return the most relevant passages'''
    
    result = retriver.invoke(query)
    
    content = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    
    return( {
        "query":query,
        "content":content,
        "metadata":metadata
    })

tools = [rag_Tool]
llm_with_tools = llm.bind_tools(tools)


In [ ]:
def chat_node(state:graph_state)->dict:
    data = state['messages']

    res = llm_with_tools.invoke(data)

    return {
        "messages":[res]
    }



tool_node = ToolNode(tools)


In [ ]:
#CREATING NODES
graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

#adding edges
graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')

chatbot = graph.compile()
chatbot

In [ ]:
res = chatbot.invoke({"messages":[HumanMessage("QUERY")]})
res['messages'][-1]